# Feature engineering

Create eight candidate features from the cleaned predictors. The train/test split already exists; feature construction uses predictors only, and all imputation is fitted inside cross-validation.

- `ca_exang` and `cp_thal` encode each pair as a distinct category. Their source codes are single digits, so `10 * first + second` preserves every combination.
- `thalach_exang` and `oldpeak_exang` are product interactions.
- `age_bin` groups ages into `<40`, `40–49`, `50–59`, `60–69`, and `70+`.
- `chol_per_age`, `bps_per_age`, and `hr_per_age` divide cholesterol, resting blood pressure, and maximum heart rate by age. A zero age produces a missing ratio for imputation.


In [6]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier


In [7]:
data_dir = Path('../data/processed')
train_df = pd.read_csv(data_dir / 'train_cleaned.csv')
test_df = pd.read_csv(data_dir / 'test_cleaned.csv')

X_train, y_train = train_df.drop(columns='target'), train_df['target']
X_test, y_test = test_df.drop(columns='target'), test_df['target']


## Build the features

The same deterministic transformation is applied to the training and test predictors. Missing `ca` or `thal` values remain missing in their pair feature and are imputed by the model pipeline.


In [ ]:
def add_features(X):
    X = X.copy()
    age = X['age'].replace(0, float('nan'))

    X['ca_exang'] = X['ca'].astype(str) + "_" + X['exang'].astype(str))
    X['thalach_exang'] = X['thalach'] * X['exang']
    X['oldpeak_exang'] = X['oldpeak'] * X['exang']
    X["cp_thal"] = (
    X["cp"].astype(str)+ "_" + X["thal"].astype(str))    
    X['age_bin'] = pd.cut(
        X['age'], bins=[float('-inf'), 40, 50, 60, 70, float('inf')],
        labels=False, right=False
    )
    X['chol_per_age'] = X['chol'] / age
    X['bps_per_age'] = X['trestbps'] / age
    X['hr_per_age'] = X['thalach'] / age
    return X


new_features = [
    'ca_exang', 'thalach_exang', 'oldpeak_exang', 'cp_thal',
    'age_bin', 'chol_per_age', 'bps_per_age', 'hr_per_age'
]
X_train_fe = add_features(X_train)
X_test_fe = add_features(X_test)

assert set(new_features).issubset(X_train_fe.columns)
assert X_train_fe.columns.equals(X_test_fe.columns)
X_train_fe[new_features].head()


,ca_exang,thalach_exang,oldpeak_exang,cp_thal,age_bin,chol_per_age,bps_per_age,hr_per_age
0,0.0,0,0.0,4_7.0,1,5.708333,2.583333,3.458333
1,0.0,0,0.0,2_3.0,2,4.763636,2.363636,2.818182
2,11.0,159,0.0,2_3.0,2,5.333333,2.444444,2.944444
3,0.0,0,0.0,2_7.0,2,5.722222,2.000000,2.888889
4,1.0,123,0.2,4_7.0,2,4.228070,2.456140,2.157895


## Compare with the original predictors

Use the same five stratified folds and decision-tree settings for both feature sets. Missing values are imputed and categories are encoded within each training fold, including the new pair categories and age bins. The held-out test set is evaluated once after the cross-validation comparison.


In [9]:
base_numeric = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
base_categorical = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
engineered_numeric = ['thalach_exang', 'oldpeak_exang', 'chol_per_age', 'bps_per_age', 'hr_per_age']
engineered_categorical = ['ca_exang', 'cp_thal', 'age_bin']


def make_tree_pipeline(numeric_cols, categorical_cols):
    preprocess = ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric_cols),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)
    ])
    return Pipeline([
        ('preprocess', preprocess),
        ('model', DecisionTreeClassifier(random_state=42))
    ])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall',
           'f1': 'f1', 'roc_auc': 'roc_auc'}
model_specs = {
    'Original': (X_train, base_numeric, base_categorical),
    'Engineered': (X_train_fe, base_numeric + engineered_numeric,
                   base_categorical + engineered_categorical)
}

results = []
for name, (features, numeric_cols, categorical_cols) in model_specs.items():
    scores = cross_validate(
        make_tree_pipeline(numeric_cols, categorical_cols), features, y_train,
        cv=cv, scoring=scoring
    )
    results.append({
        'Feature set': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1': scores['test_f1'].mean(),
        'ROC-AUC': scores['test_roc_auc'].mean(),
        'F1 Std': scores['test_f1'].std()
    })

cv_results = pd.DataFrame(results).set_index('Feature set')
cv_results.round(3)


,Accuracy,Precision,Recall,F1,ROC-AUC,F1 Std
Feature set,,,,,,
Original,0.698,0.678,0.684,0.677,0.697,0.042
Engineered,0.719,0.682,0.729,0.703,0.719,0.019


In [10]:
engineered_model = make_tree_pipeline(
    base_numeric + engineered_numeric, base_categorical + engineered_categorical
)
engineered_model.fit(X_train_fe, y_train)
test_pred = engineered_model.predict(X_test_fe)
test_prob = engineered_model.predict_proba(X_test_fe)[:, 1]

holdout_results = pd.Series({
    'Accuracy': accuracy_score(y_test, test_pred),
    'Precision': precision_score(y_test, test_pred),
    'Recall': recall_score(y_test, test_pred),
    'F1': f1_score(y_test, test_pred),
    'ROC-AUC': roc_auc_score(y_test, test_prob)
}, name='Engineered features (test)')
holdout_results.round(3)


Accuracy     0.738
Precision    0.688
Recall       0.786
F1           0.733
ROC-AUC      0.741
Name: Engineered features (test), dtype: float64

## Result

With the same decision-tree pipeline and five stratified folds, the engineered feature set increased mean F1 from **0.677** to **0.703** and mean ROC-AUC from **0.697** to **0.719**. The fitted engineered model achieved **0.733 F1** and **0.741 ROC-AUC** on the held-out test set. These are candidate features; the small dataset means the observed gains should be treated as exploratory.